# 第 1 周末练习解答 —— 技术问答解释器

## 练习目标（理念）

为展示你对 **OpenAI API** 与本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：一个技术问题（例如「这段 Python 代码在干什么？」）
- **输出**：清晰、严谨的解释
- **额外体验**：云端 GPT **流式（streaming）**显示；本地 Llama 一次返回后渲染

这是你在课程期间自己也能天天用的工具：遇到看不懂的代码，丢进来问模型。

第 2 周之后，你还能为它加上用户界面，变成更有价值的应用。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `openai.chat.completions.create(...)` |
| `messages`（system / user） | system 定导师角色；user 放具体问题 |
| 流式输出 `stream=True` | 边收边 `update_display` |
| 本地 Ollama | `ollama.chat` + `llama3.2` |

## 怎么跑

1. `.env` 里准备好 OpenAI 密钥；本机启动 Ollama 并 pull `llama3.2`
2. 从上到下运行；在「提问」格改 `question`，再分别跑 GPT 与 Llama 两格对比


In [ ]:
# ========== 导入：密钥、展示、OpenAI SDK、Ollama ==========

# 从 dotenv 导入 load_dotenv：把 .env 密钥读进环境变量，避免写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：Markdown 渲染、display、流式更新 update_display
from IPython.display import Markdown, display, update_display
# 从 openai 导入 OpenAI 客户端：调用云端 Chat Completions
from openai import OpenAI
# 导入 ollama 包：用 Python API 调本地模型（不必自己拼 HTTP）
import ollama


In [ ]:
# ========== 确保本地模型已就绪 ==========

# shell 魔法：拉取（或确认）llama3.2；模型名须与后面 MODEL_LLAMA 一致
!ollama pull llama3.2


In [ ]:
# ========== 常量：模型名字集中写在一处 ==========

# OpenAI 云端小模型 id：便宜、适合解释类问答
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：需事先 ollama pull；字符串须与本机已安装名一致
MODEL_LLAMA = 'llama3.2'


In [ ]:
# ========== 环境 + 客户端 ==========

# 加载 .env：把 OPENAI_API_KEY 等读入进程环境
load_dotenv()
# 创建 OpenAI 客户端（默认读环境变量里的密钥）
openai = OpenAI()



In [ ]:
# ========== 提问：改这里的字符串就能问新问题 ==========

# 把技术问题写在三引号字符串里；发给模型的内容保持英文（可运行 / 影响回答的字符串不翻译）
# 练习建议：换成你自己今天看不懂的一行代码，再分别跑下面 GPT / Llama 两格做对比
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""


In [ ]:
# ========== 提示词：system 定角色，user 包装问题 ==========

# system：技术导师人设；发给模型的指令保留英文
system_prompt = "You are a helpful technical tutor who answers questions about python code, software engineering, data science and LLMs"
# user：固定前缀 + 上面的 question（字符串拼接）
user_prompt = "Please give a detailed explanation to the following question: " + question


In [ ]:
# ========== messages：Chat Completions 标准消息列表 ==========

# 两条消息：先 system 定规矩，再 user 放问题
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]


In [ ]:
# ========== 路径 A：用云端 gpt-4o-mini 流式回答 ==========

# stream=True：服务端边生成边推送；返回可迭代的 chunk 流
stream = openai.chat.completions.create(model=MODEL_GPT, messages=messages,stream=True)

# 累加已收到的文本；先放一个空 Markdown 占位，拿到 display_id 便于原地更新
response = ""
display_handle = display(Markdown(""), display_id=True)
# 逐块迭代流式结果
for chunk in stream:
    # delta.content 可能是 None（某些控制块）；用 or '' 避免拼接报错
    response += chunk.choices[0].delta.content or ''
    # 去掉围栏标记，方便直接当 Markdown 渲染
    response = response.replace("```","").replace("markdown", "")
    # 用同一个 display_id 刷新内容，实现「打字机」效果
    update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
# ========== 路径 B：用本地 Llama 3.2（Ollama）一次回答 ==========

# ollama.chat：同步调用本地模型；messages 与上面 GPT 路径相同，便于对比
response = ollama.chat(model=MODEL_LLAMA, messages=messages)
# 从返回字典里取出助手回复正文
reply = response['message']['content']
# 以 Markdown 展示完整回答
display(Markdown(reply))


# 恭喜！

可以进一步改进：用下面方式交互式接收问题：

```python
my_question = input("Please enter your question:")
```

然后据此创建提示词并调用模型——同一套 `messages` + GPT / Llama 两条路径仍然适用。
